# Load Weaver

A minimal Microsoft Fabric notebook that loads an installed Weaver runtime.

`weaver build ... --to <Fabric target>` installs the runtime bundle under
`Files/_weaver/runtime` in the Lakehouse and materialises empty Delta tables.
This notebook is the load half: it reads that installed runtime and executes
each object in dependency order. It never reads the SES repository.

Keep the notebook body thin — all behaviour lives in the installed runtime.

## Parameters

In [ ]:
# Fabric injects the default Lakehouse mount; these default to it.
WEAVER_RUNTIME_ROOT = globals().get(
    "WEAVER_RUNTIME_ROOT", "/lakehouse/default/Files/_weaver/runtime"
)

# OneLake abfss root of the same Lakehouse. Spark reads and writes Delta here:
# the FUSE mount cannot host Spark Delta writes, so pass the abfss path.
WORKSPACE_ID = globals().get("WORKSPACE_ID", "<workspace-guid>")
LAKEHOUSE_ID = globals().get("LAKEHOUSE_ID", "<lakehouse-guid>")
WEAVER_SPARK_ROOT = globals().get(
    "WEAVER_SPARK_ROOT",
    f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}",
)

# The database representations to load, in order (Raw files, then Core Delta).
LOAD_TARGETS = globals().get("LOAD_TARGETS", ("Raw_Files_Fabric", "Core_Delta_Fabric"))

## Run

In [ ]:
import json
import sys
from pathlib import Path


def as_targets(value):
    if isinstance(value, str):
        return tuple(item.strip() for item in value.split(",") if item.strip())
    return tuple(value)


# The installed bundle ships its own orchestrator package; import it from there.
orchestrator_path = str(Path(WEAVER_RUNTIME_ROOT) / "_orchestrator")
if orchestrator_path not in sys.path:
    sys.path.insert(0, orchestrator_path)

from weaver_runtime.dbrep.runtime.orchestrator import load_target_runtime

runtime_root = Path(WEAVER_RUNTIME_ROOT)
reports = {
    target: load_target_runtime(
        runtime_root,
        target_filter=target,
        spark=spark,
        spark_root=WEAVER_SPARK_ROOT,
    ).to_dict()
    for target in as_targets(LOAD_TARGETS)
}

print(json.dumps(reports, indent=2, default=str))

In [ ]:
from notebookutils import mssparkutils

mssparkutils.notebook.exit("Weaver load completed")